[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jman4162/sensortwin-transformer-agent/blob/master/notebooks/05_calibration_colab.ipynb)

# SensorTwin — temperature scaling for the transformer

Runs `scripts/calibrate_transformer` (same command as `make calibration`): per seed, train
`SensorPatchTST` with its committed recipe, fit one temperature on the **validation** logits, and
measure test ECE before/after. Temperature scaling divides logits by a scalar, so predictions —
and every accuracy metric — are unchanged by construction.

The committed artifact from the 2026-07-02 run (3 seeds, generator v2): T = 0.64 ± 0.01 cuts test
ECE 0.091 ± 0.008 → 0.016 ± 0.005. Re-running here regenerates
`reports/experiment_summaries/calibration_temperature.{json,md}`. Budget: ~30-45 min/seed on a T4.

In [ ]:
# Opened from the Colab badge? Only the notebook is present — clone the public repo, then install.
import os

if not os.path.exists("sensortwin"):
    !git clone https://github.com/jman4162/sensortwin-transformer-agent.git
    %cd sensortwin-transformer-agent
%pip install -q -e ".[ml]"

In [ ]:
import torch

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
MODE = "colab_standard"
SEEDS = [0, 1, 2]
EPOCHS = 30
OUT = "reports/experiment_summaries"

In [ ]:
from scripts.calibrate_transformer import main as calibrate

calibrate(["--mode", MODE, "--seeds", *[str(s) for s in SEEDS], "--epochs", str(EPOCHS), "--out", OUT])

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

display(Markdown(Path(f"{OUT}/calibration_temperature.md").read_text()))

In [ ]:
try:
    from google.colab import files

    files.download(f"{OUT}/calibration_temperature.md")
    files.download(f"{OUT}/calibration_temperature.json")
except Exception as e:
    print("Not in Colab or download unavailable:", e)